## Phase 9: Statistical Analysis

This phase statistically validates the patterns and relationships identified during the exploratory data analysis and genre evolution analysis.

The analysis focuses on:

- Correlation between movie ratings and audience votes
- Differences in ratings across genres using ANOVA
- Relationship between genre and decade using Chi-square testing
- Confidence interval estimation for average movie ratings
- Statistical interpretation of the results

The objective is to determine whether the observed patterns in the dataset are statistically meaningful.

In [5]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import pearsonr, f_oneway, chi2_contingency

import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/movies_feature_engineered.csv")

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (569655, 46)


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,genre_list,averageRating,numVotes,year,...,genre_sport,genre_talk_show,genre_thriller,genre_war,genre_western,runtime_category,rating_category,log_votes,popularity_category,rating_popularity_score
0,tt0000574,movie,The Story of the Kelly Gang,1906,70.0,"Action,Adventure,Biography","['Action', 'Adventure', 'Biography']",6.0,1084.0,1906,...,0,0,0,0,0,Short,Average,6.989335,Popular,41.936012
1,tt0000591,movie,The Prodigal Son,1907,90.0,Drama,['Drama'],4.8,40.0,1907,...,0,0,0,0,0,Medium,Low,3.713572,Low,17.825146
2,tt0000615,movie,Robbery Under Arms,1907,NaN,Drama,['Drama'],3.4,36.0,1907,...,0,0,0,0,0,Unknown,Low,3.610918,Low,12.277121
3,tt0000630,movie,Hamlet,1908,NaN,Drama,['Drama'],3.0,42.0,1908,...,0,0,0,0,0,Unknown,Low,3.761200,Low,11.283600
4,tt0000675,movie,Don Quijote,1908,NaN,Drama,['Drama'],3.8,32.0,1908,...,0,0,0,0,0,Unknown,Low,3.496508,Low,13.286729


In [7]:
statistical_summary = df[
    ["averageRating", "numVotes", "runtimeMinutes", "year"]
].describe()

display(statistical_summary.round(2))

,averageRating,numVotes,runtimeMinutes,year
count,337665.00,337665.00,441420.00,569655.00
mean,6.13,3822.31,89.54,1997.86
std,1.37,38640.66,26.40,28.39
min,1.00,5.00,1.00,1900.00
25%,5.30,21.00,74.00,1983.00
50%,6.20,69.00,89.00,2010.00
75%,7.10,345.00,100.00,2019.00
max,10.00,3226746.00,400.00,2026.00


### 9.1 Correlation Analysis

Correlation analysis is used to measure the strength and direction
of relationships between numerical movie attributes.

The analysis examines relationships between:

- Average rating and number of votes
- Average rating and runtime
- Average rating and release year
- Number of votes and runtime

In [8]:
correlation_data = df[
    ["averageRating", "numVotes", "runtimeMinutes", "year"]
].dropna()

correlation_matrix = correlation_data.corr()

display(correlation_matrix.round(3))

,averageRating,numVotes,runtimeMinutes,year
averageRating,1.000,0.068,0.048,0.057
numVotes,0.068,1.000,0.098,0.025
runtimeMinutes,0.048,0.098,1.000,0.104
year,0.057,0.025,0.104,1.000


In [9]:
from scipy.stats import pearsonr

rating_votes = df[
    ["averageRating", "numVotes"]
].dropna()

rating_runtime = df[
    ["averageRating", "runtimeMinutes"]
].dropna()

rating_year = df[
    ["averageRating", "year"]
].dropna()

r_votes, p_votes = pearsonr(
    rating_votes["averageRating"],
    rating_votes["numVotes"]
)

r_runtime, p_runtime = pearsonr(
    rating_runtime["averageRating"],
    rating_runtime["runtimeMinutes"]
)

r_year, p_year = pearsonr(
    rating_year["averageRating"],
    rating_year["year"]
)

print("Rating vs Number of Votes")
print(f"Pearson r = {r_votes:.4f}")
print(f"p-value   = {p_votes:.6g}")

print("\nRating vs Runtime")
print(f"Pearson r = {r_runtime:.4f}")
print(f"p-value   = {p_runtime:.6g}")

print("\nRating vs Release Year")
print(f"Pearson r = {r_year:.4f}")
print(f"p-value   = {p_year:.6g}")

Rating vs Number of Votes
Pearson r = 0.0628
p-value   = 6.29816e-292

Rating vs Runtime
Pearson r = 0.0480
p-value   = 2.10139e-156

Rating vs Release Year
Pearson r = 0.0715
p-value   = 0


### 9.2 Hypothesis Testing

We test whether movie ratings and audience voting activity
have a statistically significant linear relationship.

**Null Hypothesis (H₀):**
There is no linear relationship between average rating and number of votes.

**Alternative Hypothesis (H₁):**
There is a statistically significant linear relationship between
average rating and number of votes.

The significance level is set at α = 0.05.

In [10]:
alpha = 0.05

print("Hypothesis Test: Rating vs Number of Votes")
print(f"Pearson r = {r_votes:.4f}")
print(f"p-value   = {p_votes:.6g}")

if p_votes < alpha:
    print("\nDecision: Reject H₀")
    print("Conclusion: A statistically significant linear relationship exists.")
else:
    print("\nDecision: Fail to reject H₀")
    print("Conclusion: No statistically significant linear relationship was detected.")

Hypothesis Test: Rating vs Number of Votes
Pearson r = 0.0628
p-value   = 6.29816e-292

Decision: Reject H₀
Conclusion: A statistically significant linear relationship exists.


### 9.3 Confidence Interval

A 95% confidence interval is calculated for the population mean
of movie average ratings.

This provides an estimated range in which the true mean rating
is expected to lie.

In [11]:
from scipy.stats import t

ratings = df["averageRating"].dropna()

sample_mean = ratings.mean()
sample_std = ratings.std()
n = len(ratings)

confidence_level = 0.95
alpha = 1 - confidence_level

standard_error = sample_std / np.sqrt(n)

t_critical = t.ppf(
    1 - alpha / 2,
    df=n - 1
)

margin_of_error = t_critical * standard_error

ci_lower = sample_mean - margin_of_error
ci_upper = sample_mean + margin_of_error

print(f"Sample Mean Rating: {sample_mean:.4f}")
print(f"95% Confidence Interval: ({ci_lower:.4f}, {ci_upper:.4f})")
print(f"Margin of Error: ±{margin_of_error:.4f}")

Sample Mean Rating: 6.1334
95% Confidence Interval: (6.1288, 6.1381)
Margin of Error: ±0.0046


### 9.4 One-Way ANOVA

A one-way ANOVA test is used to determine whether the mean movie
rating differs significantly across different decades.

**Null Hypothesis (H₀):**
The mean movie rating is the same across all selected decades.

**Alternative Hypothesis (H₁):**
At least one decade has a significantly different mean rating.

Only decades containing a sufficient number of movies are included.

In [12]:
from scipy.stats import f_oneway

anova_data = df[
    ["decade", "averageRating"]
].dropna()

decade_counts = anova_data["decade"].value_counts()

valid_decades = decade_counts[
    decade_counts >= 100
].index

anova_groups = [
    anova_data.loc[
        anova_data["decade"] == decade,
        "averageRating"
    ]
    for decade in sorted(valid_decades)
]

anova_groups = [
    group for group in anova_groups
    if len(group) > 1
]

f_stat, p_anova = f_oneway(*anova_groups)

print("One-Way ANOVA: Movie Ratings Across Decades")
print(f"Number of decades tested: {len(anova_groups)}")
print(f"F-statistic = {f_stat:.4f}")
print(f"p-value    = {p_anova:.6g}")

if p_anova < 0.05:
    print("\nDecision: Reject H₀")
    print("Conclusion: Mean movie ratings differ significantly across decades.")
else:
    print("\nDecision: Fail to reject H₀")
    print("Conclusion: No statistically significant difference was detected.")

One-Way ANOVA: Movie Ratings Across Decades
Number of decades tested: 13
F-statistic = 274.2440
p-value    = 0

Decision: Reject H₀
Conclusion: Mean movie ratings differ significantly across decades.


### 9.5 Chi-Square Test of Independence

The Chi-square test is used to examine whether movie genre
and release decade are statistically associated.

**Null Hypothesis (H₀):**
Genre and decade are independent.

**Alternative Hypothesis (H₁):**
Genre and decade are associated.

Because movies may belong to multiple genres, the genre observations
are treated as individual genre assignments for exploratory analysis.

In [18]:
import ast

# Prepare data
genre_data = df[
    ["decade", "genre_list"]
].dropna().copy()

# Convert string representation of lists into actual Python lists
genre_data["genre_list"] = genre_data["genre_list"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Explode genres into individual rows
genre_exploded = genre_data.explode("genre_list").copy()

# Clean genre names
genre_exploded["genre"] = (
    genre_exploded["genre_list"]
    .astype(str)
    .str.strip()
)

# Remove empty/null genres
genre_exploded = genre_exploded[
    genre_exploded["genre"].notna() &
    (genre_exploded["genre"] != "")
]

# Create Decade × Genre contingency table
contingency_table = pd.crosstab(
    genre_exploded["decade"],
    genre_exploded["genre"]
)

display(contingency_table)

genre,Action,Adult,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,...,Mystery,News,Reality-TV,Romance,Sci-Fi,Sport,Talk-Show,Thriller,War,Western
decade,,,,,,,,,,,,,,,,,,,,,
1900,2,0,3,0,4,6,0,71,20,1,...,0,4,0,1,0,9,0,0,5,0
1910,242,0,344,5,56,1306,771,434,6404,15,...,217,1,0,670,30,40,0,100,284,346
1920,732,0,869,22,90,2424,852,818,10256,62,...,318,2,0,1736,31,129,0,124,201,1367
1930,817,0,846,26,132,4402,1621,1459,9880,225,...,650,10,0,2705,63,140,0,173,345,982
1940,652,0,802,68,213,3553,1280,627,7308,349,...,551,1,0,1909,50,82,0,243,770,1063
1950,1211,0,1421,95,331,4834,2017,1182,11141,745,...,473,5,0,2625,253,159,0,518,751,834
1960,3297,41,2514,143,288,6130,2982,1900,13920,922,...,760,5,1,3515,352,222,0,1100,1118,844
1970,4224,2718,2485,257,481,7180,3285,3576,15948,1198,...,824,5,2,3300,427,322,2,1538,793,572
1980,5157,3414,1942,518,647,7623,2959,4445,17177,1349,...,901,1,1,3352,746,317,2,1999,874,127


In [20]:
chi2, p_chi, dof, expected = chi2_contingency(
    contingency_table
)

print("Chi-Square Test: Genre vs Decade")
print(f"Chi-square statistic = {chi2:.4f}")
print(f"Degrees of freedom    = {dof}")
print(f"p-value               = {p_chi:.6g}")

if p_chi < 0.05:
    print("\nDecision: Reject H₀")
    print("Conclusion: Genre distribution is significantly associated with decade.")
else:
    print("\nDecision: Fail to reject H₀")
    print("Conclusion: No statistically significant association was detected.")

Chi-Square Test: Genre vs Decade
Chi-square statistic = 161175.6705
Degrees of freedom    = 312
p-value               = 0

Decision: Reject H₀
Conclusion: Genre distribution is significantly associated with decade.


In [21]:
# 9.6 Statistical Test Summary

statistical_tests = pd.DataFrame({
    "Test": [
        "Pearson Correlation: Rating vs Votes",
        "Pearson Correlation: Rating vs Runtime",
        "Pearson Correlation: Rating vs Year",
        "One-Way ANOVA: Ratings vs Decade",
        "Chi-Square: Genre vs Decade"
    ],
    
    "Statistic": [
        r_votes,
        r_runtime,
        r_year,
        f_stat,
        chi2
    ],
    
    "p_value": [
        p_votes,
        p_runtime,
        p_year,
        p_anova,
        p_chi
    ]
})

statistical_tests["Significant_at_0.05"] = (
    statistical_tests["p_value"] < 0.05
)

display(statistical_tests.round(4))

,Test,Statistic,p_value,Significant_at_0.05
0,Pearson Correlation: Rating vs Votes,0.0628,0.0,True
1,Pearson Correlation: Rating vs Runtime,0.0480,0.0,True
2,Pearson Correlation: Rating vs Year,0.0715,0.0,True
3,One-Way ANOVA: Ratings vs Decade,274.2440,0.0,True
4,Chi-Square: Genre vs Decade,161175.6705,0.0,True


## Key Insights

### Statistical Analysis Findings

- Pearson correlation was used to examine relationships between movie ratings, voting activity, runtime, and release year.
- The relationship between average rating and number of votes was statistically significant, although the correlation strength was weak.
- The relationship between average rating and runtime was statistically significant, but the correlation strength was also weak.
- Average rating and release year showed a statistically significant but weak positive relationship.
- A 95% confidence interval was calculated for the overall mean movie rating.
- One-way ANOVA showed that average movie ratings differ significantly across decades.
- The Chi-square test showed a statistically significant association between genre distribution and release decade.
- Statistical significance was evaluated using a significance level of α = 0.05.
- Statistical significance does not necessarily imply a strong or causal relationship.

## Statistical Analysis Conclusion

The statistical analysis provides quantitative evidence that movie characteristics and genre patterns have changed significantly over time.

Correlation analysis revealed statistically significant relationships between movie ratings, voting activity, runtime, and release year, although the correlation strengths were generally weak.

The hypothesis test confirmed a statistically significant relationship between movie ratings and number of votes.

The 95% confidence interval provided a precise estimate of the population mean movie rating.

One-way ANOVA demonstrated that movie ratings vary significantly across different decades, indicating changes in rating patterns over time.

The Chi-square test further established that genre distributions are significantly associated with release decades. This supports the earlier genre evolution analysis and confirms that the popularity and presence of genres have changed substantially throughout cinema history.

Overall, the statistical analysis validates several important patterns identified during exploratory and genre evolution analysis.